# TrackViewer Visualization of Filtered STRs and Functional Annotations

This notebook uses the [trackViewer](https://bioconductor.org/packages/trackViewer) (Bioconductor) R package to visualize the **8 filtered STRs** from `Relatorio_STR_Final_Integral.pdf` together with their functional annotations.

**Inputs**
- Local project data: STR coordinates / residual / scRNA-seq expression
- External tracks downloaded by `7.4.3.1_download_external_tracks.sh` into `external_tracks/`

**Reference genome:** hg38 (GRCh38).

In [ ]:
suppressPackageStartupMessages({
  library(trackViewer)
  library(GenomicRanges)
  library(rtracklayer)
  library(TxDb.Hsapiens.UCSC.hg38.knownGene)
  library(readr)
})

cat('trackViewer loaded OK\n')

## 2. Define the 8 filtered STRs

The 8 STR loci are loaded from the unified scRNA-seq overlap file generated in step 7.3.1
(`../7.3_STRs_filter/results/STR_vs_scRNA_overlap_unified.csv`), taking one row per locus.
Genomic coordinates (`start0` 0-based, `end`) are taken from `STR_variants_UCSC_track.bed`
(shipped with this notebook), falling back to `start0 + nchar(motif) * copy` if the BED is absent.


In [ ]:
# 2. Define the 8 filtered STRs from the unified overlap CSV + BED coordinates
variants_file <- '../7.3_STRs_filter/results/STR_vs_scRNA_overlap_unified.csv'
bed_file <- 'STR_variants_UCSC_track.bed'

if (file.exists(variants_file)) {
  # one row per STR locus (drop duplicated cell_type rows)
  ov <- read_csv(variants_file, show_col_types = FALSE)
  variants <- ov[!duplicated(ov$STRs_ID), ]

  # STRs_ID encodes chr:pos:motif:copy  (pos = 1-based start)
  id_parts <- strsplit(variants$STRs_ID, ':', fixed = TRUE)
  variants$chr    <- vapply(id_parts, function(x) x[1], character(1))
  variants$start1 <- as.integer(vapply(id_parts, function(x) x[2], character(1)))
  variants$motif  <- vapply(id_parts, function(x) x[3], character(1))
  variants$copy   <- as.integer(vapply(id_parts, function(x) x[4], character(1)))
  variants$start0 <- variants$start1 - 1
  variants$gene   <- variants$gene_name
  variants$allele2 <- variants$allele2_est
  variants$group  <- ifelse(tolower(variants$group) == 'case', 'Case', 'Control')

  # BED coordinates for start0 / end (0-based start, 1-based-style end)
  if (file.exists(bed_file)) {
    bed <- read.delim(bed_file, comment.char = '#', header = FALSE, fill = TRUE,
                      col.names = c('chr', 'start0', 'end', 'name'))
    bed <- bed[grepl('^chr', bed$chr), ]
    bed$STRs_ID <- sub('^[^_]+_', '', bed$name)  # name = <gene>_<STRs_ID>
    bed <- bed[bed$STRs_ID %in% variants$STRs_ID, c('STRs_ID', 'end')]
    variants$end <- bed$end[match(variants$STRs_ID, bed$STRs_ID)]
    if (anyNA(variants$end)) {
      warning('Some loci missing from BED; estimating end from motif length')
      na <- is.na(variants$end)
      variants$end[na] <- variants$start0[na] + nchar(variants$motif[na]) * variants$copy[na]
    }
  } else {
    cat('BED file not found; estimating end from motif length * copy\n')
    variants$end <- variants$start0 + nchar(variants$motif) * variants$copy
  }

  cat('Loaded', nrow(variants), 'STR loci from', variants_file, '\n')
} else {
  stop('Overlap CSV not found: ', variants_file)
}

print(variants[, c('STRs_ID', 'gene', 'abs_res', 'allele2', 'group', 'chr', 'start0', 'end')])

# GRanges of the STRs (1-based start)
gr_strs <- GRanges(seqnames = variants$chr,
                   ranges = IRanges(start = variants$start0 + 1, end = variants$end),
                   strand = '*',
                   STRs_ID = variants$STRs_ID,
                   gene = variants$gene,
                   abs_res = variants$abs_res,
                   allele2 = variants$allele2,
                   group = variants$group,
                   motif = variants$motif)
names(gr_strs) <- variants$gene
gr_strs

## 3. Load external tracks

Files are expected in `external_tracks/` (created by the download script). Each track is a helper function that returns a `trackViewer` feature track (importing bigWig/BED via `rtracklayer::import`).

In [ ]:
ext_dir <- 'external_tracks'
stopifnot(dir.exists(ext_dir))

# Window covering all 8 loci (+/- 20 kb) so we import only what is plotted
# instead of the whole-genome bigWig (avoids multi-GB in-memory GRanges).
win_all <- if (exists('gr_strs') && length(gr_strs) > 0) {
  reduce(resize(gr_strs, width = 40000, fix = 'center'))
} else {
  NULL  # fall back to whole-genome import if cell 3 did not produce loci
}

# Track building helpers -----------------------------------------------------
# class is 'track' (lowercase); color lives in trackStyle (setTrackStyleParam)

# quick magic-byte sanity checks to avoid importing error pages saved as .bw
is_bigwig <- function(path) {
  con <- file(path, 'rb'); on.exit(close(con))
  magic <- readBin(con, 'raw', n = 4)
  length(magic) == 4 && (
    identical(magic, as.raw(c(0x26, 0xfc, 0x8f, 0x88))) ||  # bigWig LE
    identical(magic, as.raw(c(0x88, 0x8f, 0xfc, 0x26)))     # bigWig BE
  )
}

is_text_error <- function(path) {
  con <- file(path, 'rb'); on.exit(close(con))
  magic <- readBin(con, 'raw', n = 2)
  if (length(magic) == 2 && identical(magic, as.raw(c(0x1f, 0x8b)))) return(FALSE)
  head <- readChar(con, nchars = 200, useBytes = TRUE)
  grepl('<html|<!DOCTYPE', head, ignore.case = TRUE, useBytes = TRUE)
}

import_bw <- function(path, name, color, win = NULL) {
  if (!file.exists(path)) {
    warning('Missing file: ', path)
    return(NULL)
  }
  if (!is_bigwig(path)) {
    warning('Not a valid bigWig (magic bytes), skipping: ', path)
    return(NULL)
  }
  gr <- rtracklayer::import(path, which = win)
  if (length(gr) == 0) return(NULL)
  if (is.null(gr$score)) mcols(gr)$score <- 0
  tr <- new("track", dat = gr, type = "data", format = "BigWig", name = name)
  setTrackStyleParam(tr, "color", color)
  tr
}

import_bed <- function(path, name, color, win = NULL) {
  if (!file.exists(path)) {
    warning('Missing file: ', path)
    return(NULL)
  }
  if (is_text_error(path)) {
    warning('Looks like an error/HTML page, skipping: ', path)
    return(NULL)
  }
  gr <- rtracklayer::import(path, format = 'BED', which = win)
  if (length(gr) == 0) return(NULL)
  if (is.null(gr$score)) mcols(gr)$score <- 1
  tr <- new("track", dat = gr, type = "data", format = "BED", name = name)
  setTrackStyleParam(tr, "color", color)
  tr
}

# Individual annotation files -----------------------------------------------
tracks_external <- list(
  CTCF   = import_bw(file.path(ext_dir, 'CTCF_ENCFF910VLV.bw'),       'CTCF ChIP-seq',    '#D7301F', win_all),
  DNase  = import_bw(file.path(ext_dir, 'DNase_brain.bw'),            'DNase',            '#E08214', win_all),
  H3K27ac= import_bw(file.path(ext_dir, 'H3K27ac_brain.bw'),          'H3K27ac',          '#8073AC', win_all),
  RemapD = import_bw(file.path(ext_dir, 'remap2022_density_hg38.bw'), 'ReMap Density',    '#4575B4', win_all)
)

# ReMap TF peak tracks (any available) - keyed with a _peaks suffix so the
# CTCF ReMap peaks do not overwrite the CTCF ChIP-seq signal track.
tf_names <- c('CTCF','REST','KLF9','GATA2','ZNF384','MNT')
tf_files <- c(
  'remap2022_ctcf_all_macs2_hg38_v1_0.bed.gz',
  'remap2022_rest_all_macs2_hg38_v1_0.bed.gz',
  'remap2022_klf9_all_macs2_hg38_v1_0.bed.gz',
  'remap2022_gata2_all_macs2_hg38_v1_0.bed.gz',
  'remap2022_znf384_all_macs2_hg38_v1_0.bed.gz',
  'remap2022_mnt_all_macs2_hg38_v1_0.bed.gz'
)
for (i in seq_along(tf_names)) {
  tr <- import_bed(file.path(ext_dir, tf_files[i]), tf_names[i], '#238B45', win_all)
  if (!is.null(tr)) tracks_external[[paste0(tf_names[i], '_peaks')]] <- tr
}

cat('External tracks loaded:', sum(!vapply(tracks_external, is.null, logical(1))), '/', length(tracks_external), '
')

## 4. Load local project data (scRNA-seq expression)

Optional overlay: LogFC per cell type from the unified overlap CSV (`../7.3_STRs_filter/results/STR_vs_scRNA_overlap_unified.csv`), shipped in this repo, used to color/adjust the STR features.


In [ ]:
scrna_file <- '../7.3_STRs_filter/results/STR_vs_scRNA_overlap_unified.csv'
if (file.exists(scrna_file)) {
  scrna <- read_csv(scrna_file, show_col_types = FALSE)
  scrna <- scrna[!is.na(scrna$LogFC) & !is.na(scrna$STRs_ID), ]
  cat('scRNA overlap rows:', nrow(scrna), '\n')
  print(unique(scrna[, c('gene_name','STRs_ID','source_tissue','LogFC')]))
} else {
  cat('scRNA overlap file not found; skipping overlay\n')
  scrna <- NULL
}

## 5. Gene model track

Build a gene track per locus from the TxDb (hg38 knownGene).

In [ ]:
txdb <- TxDb.Hsapiens.UCSC.hg38.knownGene

# geneModelFromTxdb returns a LIST of transcript tracks (one per transcript)
gene_track_for <- function(gr_variant) {
  # extend window a bit to capture the gene context
  win <- gr_variant
  start(win) <- start(win) - 20000
  end(win)   <- end(win)   + 20000
  geneModelFromTxdb(txdb, gr = win)
}

cat('Gene model helper ready\n')

## 6. Variant track with annotations

Each STR is drawn as a feature with height scaled by absolute residual and colored by group.

In [ ]:
make_str_track <- function(gr_variant) {
  gr <- gr_variant
  mcols(gr)$score <- 1
  tr <- new("track", dat = gr, type = "data", format = "BED", name = gr$gene)
  setTrackStyleParam(tr, "color", ifelse(gr$group == 'Case', '#C62828', '#1565C0'))
  setTrackStyleParam(tr, "height", 0.4)
  tr
}

cat('Variant track helper ready\n')

## 7. scRNA-seq expression overlay (optional track)

Creates a data track with LogFC values per STR locus (mean across cell types).

In [ ]:
make_scrna_track <- function(scrna, gr_variant) {
  if (is.null(scrna)) return(NULL)
  sub <- scrna[scrna$STRs_ID == gr_variant$STRs_ID, ]
  if (nrow(sub) == 0) return(NULL)
  gr <- gr_variant
  mcols(gr)$score <- mean(sub$LogFC, na.rm = TRUE)
  tr <- new("track", dat = gr, type = "data", format = "BED",
            name = paste0(gr_variant$gene, ' LogFC'))
  setTrackStyleParam(tr, "color", '#6A51A3')
  tr
}

cat('scRNA overlay helper ready\n')

## 8. Render one figure per STR

For each of the 8 loci, `viewTracks` combines: gene model, external functional tracks, scRNA LogFC and the STR variant. Figures are saved as PNG (and optionally PDF).

In [ ]:
dir.create('results', showWarnings = FALSE)

render_variant <- function(gr_variant, track_list, out_png) {
  png(out_png, width = 1400, height = 900, res = 150)
  viewTracks(track_list, gr = gr_variant,
             viewerStyle = trackViewerStyle(),
             autoOptimizeStyle = TRUE)
  dev.off()
  cat('saved:', out_png, '\n')
}

track_list_names <- names(tracks_external)

for (i in seq_len(nrow(variants))) {
  gr_v <- gr_strs[i]

  trackList <- list()
  trackNames <- character(0)

  # gene model (list of transcript tracks)
  gt <- tryCatch(gene_track_for(gr_v), error = function(e) NULL)
  if (!is.null(gt)) {
    trackList <- c(trackList, gt)
    trackNames <- c(trackNames, names(gt))
  }

  # external functional tracks
  for (nm in track_list_names) {
    tr <- tracks_external[[nm]]
    if (!is.null(tr)) {
      # subset track to a window around the variant (faster, cleaner)
      win <- resize(gr_v, width = 40000, fix = 'center')
      tr_sub <- tr
      tr_sub@dat <- subsetByOverlaps(tr@dat, win)
      trackList[[length(trackList) + 1]] <- tr_sub
      trackNames <- c(trackNames, nm)
    }
  }

  # scRNA LogFC overlay
  sct <- make_scrna_track(scrna, gr_v)
  if (!is.null(sct)) {
    trackList[[length(trackList) + 1]] <- sct
    trackNames <- c(trackNames, paste0(gr_v$gene, ' LogFC'))
  }

  # STR variant
  trackList[[length(trackList) + 1]] <- make_str_track(gr_v)
  trackNames <- c(trackNames, gr_v$gene)
  names(trackList) <- trackNames

  out_png <- sprintf('results/trackviewer_%s.png', gr_v$gene)
  render_variant(gr_v, trackList, out_png)
}

cat('\nAll variant figures generated under results/\n')

## 9. Combined panel (optional)

All 8 loci can be combined into a single `browseTracks` interactive page.

In [ ]:
if (interactive()) {
  # collect tracks for all variants and open interactive browser
  browseTracks(trackList)  # last trackList of the loop
}
cat('Done.\n')